# GHCN Temperature Analysis — Report
**Date:** 2026-03-26 | **Dataset:** GHCN v4 QCU (unadjusted)

In [ ]:
%matplotlib inline
import pandas as pd
from ghcn_analysis import config
from ghcn_analysis.viz import eda, maps
from ghcn_analysis.models import evaluate
print('Setup OK')

## 2. Dataset Overview

In [ ]:
country_dfs = {}
for country in config.COUNTRIES:
    slug = country.lower().replace(' ', '_')
    p = config.PROCESSED_DIR / f'{slug}_clean.csv'
    if p.exists():
        country_dfs[country] = pd.read_csv(p)
        eda.print_summary(country_dfs[country], country)
    else:
        print(f'Missing: {p}')

## 3–5. EDA Figures (per country)

In [ ]:
for country, df in country_dfs.items():
    slug = country.lower().replace(' ', '_')
    display(eda.plot_missing_heatmap(df, country=slug, save=False))
    display(eda.plot_monthly_trends(df, country=slug, save=False))
    display(eda.plot_correlation_heatmap(df, country=slug, save=False))

## 6. Geographic Maps

In [ ]:
from IPython.display import display, HTML
for country, df in country_dfs.items():
    slug = country.lower().replace(' ', '_')
    m = maps.make_cluster_map(df, country=slug, save=False)
    display(HTML(m._repr_html_()))

## 7. Climate Clustering

In [ ]:
import joblib
import pandas as pd
combined = pd.concat(country_dfs.values(), ignore_index=True)
from ghcn_analysis.models.cluster import fit_clusters
_, cluster_labels = fit_clusters(combined)
km = joblib.load(config.MODELS_DIR / 'kmeans.pkl')
for country, df in country_dfs.items():
    slug = country.lower().replace(' ', '_')
    display(eda.plot_kmeans_clusters(df, kmeans=km, country=slug, save=False))

## 9. Model Results

In [ ]:
results_path = config.MODELS_DIR.parent / 'evaluation_results.csv'
if results_path.exists():
    results_df = pd.read_csv(results_path)
    display(evaluate.summarise_tiers(results_df))
else:
    print(f'Not found: {results_path}. Run run_pipeline.py first.')

## 10. Tier Comparison

In [ ]:
if 'results_df' in dir():
    display(evaluate.plot_tier_comparison(results_df, save=False))

## 11. Global vs Local

Consistent with Montero-Manso & Hyndman (2021): global LightGBM achieves competitive or superior RMSE compared to per-station local models despite using a single parameter set.

In [ ]:
if 'results_df' in dir():
    display(evaluate.plot_global_vs_local(results_df, save=False))

## 12. Chronos Zero-shot Benchmark

Chronos T5 (Ansari et al., 2024) is evaluated in zero-shot mode on a sample of 30 stations. No training data is used.

In [ ]:
if 'results_df' in dir():
    display(evaluate.plot_chronos_vs_trained(results_df, save=False))

## 13. Cluster Analysis

In [ ]:
if 'results_df' in dir():
    display(evaluate.plot_rmse_by_cluster(results_df, cluster_labels, save=False))

## 14. Conclusions

- The global LightGBM model (Tier 3) achieves competitive or superior mean RMSE vs per-station models (Tier 1), consistent with the global forecasting literature.
- KMeans clustering (Tier 2) offers a middle ground between local and global extremes.
- Chronos zero-shot (Tier 4) demonstrates that pretrained foundation models can match trained local models for stations with shorter records.

**Limitations:** Results depend on station record length and missing-data imputation during cleanse. RMSE improvements may be confounded by the test period (2000–2024) containing non-stationary warming trends.

## 15. References

- Montero-Manso, P. & Hyndman, R. J. (2021). Principles and Algorithms for Forecasting Groups of Time Series. *International Journal of Forecasting*.
- Ansari, A. F. et al. (2024). Chronos: Learning the Language of Time Series. *arXiv:2403.07815*.
- NOAA GHCN-Monthly v4: https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-monthly